In [ ]:
# === DATA FUSION : historique + nouvelles donnees Supabase ===
import subprocess
import sys
import os
import pandas as pd

DATASETS_DIR  = r'C:\Users\casir\Desktop\CRYPTOS\data\datasets'
HIST_PATH     = os.path.join(DATASETS_DIR, 'ml_dataset_final.csv')
NEW_PATH      = os.path.join(DATASETS_DIR, 'ml_dataset_new.csv')
COMBINED_PATH = os.path.join(DATASETS_DIR, 'ml_dataset_combined.csv')
EXPORT_SCRIPT = r'C:\Users\casir\Desktop\CRYPTOS\repo\scripts\export_new_data.py'

# 1) Historique
print('Chargement du dataset historique...')
df_hist = pd.read_csv(HIST_PATH)
df_hist['candle_time'] = pd.to_datetime(df_hist['candle_time'], utc=True)
if 'snapshot_count' in df_hist.columns:
    df_hist = df_hist.drop(columns=['snapshot_count'])
print(f'  Historique : {len(df_hist):,} lignes')

# 2) Nouvelles donnees via export_new_data.py
print('Lancement de export_new_data.py...')
res = subprocess.run([sys.executable, EXPORT_SCRIPT], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('STDERR:', res.stderr)
    raise RuntimeError('export_new_data.py a echoue')

df_new = pd.read_csv(NEW_PATH)
df_new['candle_time'] = pd.to_datetime(df_new['candle_time'], utc=True)
if 'snapshot_count' in df_new.columns:
    df_new = df_new.drop(columns=['snapshot_count'])
print(f'  Nouvelles  : {len(df_new):,} lignes')

# 3) Concatenation + dedup + tri
df_combined = pd.concat([df_hist, df_new], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset=['symbol', 'candle_time'], keep='last')
df_combined = df_combined.sort_values(['symbol', 'candle_time']).reset_index(drop=True)

# 4) Resume
print()
print(f'Historique : {len(df_hist):,} lignes')
print(f'Nouvelles  : {len(df_new):,} lignes')
print(f'Total      : {len(df_combined):,} lignes (apres dedup)')
print(f'Periode    : {df_combined["candle_time"].min()} -> {df_combined["candle_time"].max()}')
print()
print('Lignes par symbol :')
print(df_combined.groupby('symbol').size().to_string())

# 5) Sauvegarde
df_combined.to_csv(COMBINED_PATH, index=False)
print(f'\nSauvegarde -> {COMBINED_PATH}')


In [ ]:
# === CHARGEMENT DONNEES MULTI-TIMEFRAME + FEAR & GREED ===
import numpy as np

# OHLCV 1h propre
df_1h = pd.read_csv(r'C:\Users\casir\Desktop\CRYPTOS\data\datasets\ohlcv_1h_clean.csv')
df_1h['candle_time'] = pd.to_datetime(df_1h['candle_time'], utc=True)

# OHLCV 4h propre
df_4h = pd.read_csv(r'C:\Users\casir\Desktop\CRYPTOS\data\datasets\ohlcv_4h_clean.csv')
df_4h['candle_time'] = pd.to_datetime(df_4h['candle_time'], utc=True)

# Fear & Greed historique (2018 -> aujourd'hui, daily)
df_fg = pd.read_csv(r'C:\Users\casir\Desktop\CRYPTOS\data\datasets\fear_greed_historical.csv')
df_fg['date'] = pd.to_datetime(df_fg['date'], utc=True)
df_fg = df_fg.sort_values('date').reset_index(drop=True)

print(f'1h  : {len(df_1h):,} bougies | {df_1h.candle_time.min().date()} -> {df_1h.candle_time.max().date()}')
print(f'4h  : {len(df_4h):,} bougies | {df_4h.candle_time.min().date()} -> {df_4h.candle_time.max().date()}')
print(f'F&G : {len(df_fg):,} points  | {df_fg.date.min().date()} -> {df_fg.date.max().date()}')


# 01 — Feature Engineering
**Savvy Crypto Agent — Jedha Bootcamp Final Project**

Dataset : `ml_dataset_final.csv` — 517 509 bougies 5m — 10 symboles — Juillet 2025 → Avril 2026

**Règle d'or : tout l'EDA et l'entraînement se font sur le TRAIN set (80%). Le TEST set (20%) ne sera touché qu'au notebook 05.**

## 0. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Style ---
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

# --- Chemins ---
DATA_PATH   = r'C:\Users\casir\Desktop\CRYPTOS\data\datasets\ml_dataset_combined.csv'
OUTPUT_PATH = "C:\\Users\\casir\\Desktop\\CRYPTOS\\data\\processed\\"

# --- Split ---
SPLIT_DATE   = '2026-02-10'
SYMBOLS_FULL = ['ADA-EUR', 'AVAX-EUR', 'BCH-EUR', 'DOT-EUR', 'LINK-EUR', 'LTC-EUR', 'SOL-EUR']
SYMBOLS_ALL  = SYMBOLS_FULL + ['BTC-EUR', 'ETH-EUR', 'XRP-EUR']

# ═══════════════════════════════════════════════════════
# PARAMÈTRES DE TRADE — modifie uniquement ces 4 lignes
TP_PCT   = 0.0095   # Take Profit brut  (0.010 = 1.0%)
SL_PCT   = 0.005   # Stop Loss brut    (0.005 = 0.5%)
FEES_PCT = 0.001   # Frais gas A/R     (0.001 = 0.1%)
HORIZON  = 12      # Bougies max       (12 x 5min = 1h)
# ═══════════════════════════════════════════════════════

# Calculs automatiques
TP_NET    = TP_PCT   - FEES_PCT
SL_NET    = SL_PCT   + FEES_PCT
BREAKEVEN = SL_NET   / (TP_NET + SL_NET)

print(f'TP brut        : {TP_PCT*100:.2f}%  →  net : {TP_NET*100:.2f}%')
print(f'SL brut        : {SL_PCT*100:.2f}%  →  net : {SL_NET*100:.2f}%')
print(f'Frais gas A/R  : {FEES_PCT*100:.3f}%')
print(f'Horizon max    : {HORIZON} bougies ({HORIZON*5} min)')
print(f'Seuil rentabilité : {BREAKEVEN*100:.1f}%')
print(f'Ratio R/R         : 1 : {TP_NET/SL_NET:.2f}')

## 1. Chargement & split train/test

In [ ]:
df = pd.read_csv(DATA_PATH)
df['candle_time'] = pd.to_datetime(df['candle_time'], utc=True)
df = df.sort_values(['symbol', 'candle_time']).reset_index(drop=True)

print(f'Dataset total : {len(df):,} bougies')
print(f'Période       : {df["candle_time"].min().date()} → {df["candle_time"].max().date()}')
print(f'Symboles      : {sorted(df["symbol"].unique())}')
print()

# Split chronologique — jamais de shuffle sur une série temporelle
split_ts = pd.Timestamp(SPLIT_DATE, tz='UTC')
train = df[df['candle_time'] < split_ts].copy()
test  = df[df['candle_time'] >= split_ts].copy()

print(f'TRAIN : {len(train):,} bougies ({train["candle_time"].min().date()} → {train["candle_time"].max().date()})')
print(f'TEST  : {len(test):,}  bougies ({test["candle_time"].min().date()} → {test["candle_time"].max().date()})')
print(f'Ratio : {len(train)/len(df)*100:.1f}% / {len(test)/len(df)*100:.1f}%')

## 2. Vérification qualité du dataset

In [ ]:
print('=== NULLS ===')
print(train.isnull().sum())
print()

print('=== BOUGIES PAR SYMBOLE (TRAIN) ===')
for sym, g in train.groupby('symbol'):
    gaps = g['candle_time'].diff().dt.total_seconds() / 60
    big_gaps = (gaps > 10).sum()  # gaps > 10 min
    print(f'  {sym:12s}: {len(g):6,} bougies | gaps > 10min: {big_gaps}')

print()
print('=== SNAPSHOT COUNT DISTRIBUTION ===')
if 'snapshot_count' in train.columns:
    print(train['snapshot_count'].value_counts().sort_index().head(10))
    print(f'  Bougies avec snapshot_count=1 (potentiellement bruyantes): {(train["snapshot_count"]==1).sum():,}')
else:
    print('  snapshot_count absent du dataset combiné (normal)')

## 3. Feature Engineering

On calcule tous les indicateurs **symbol par symbol** pour éviter les contaminations cross-symboles.

Règle absolue : on ne regarde jamais le futur. Toutes les features sont calculées uniquement avec des données passées (`.shift(1)` si nécessaire).

In [ ]:
def compute_rsi(series, period=14):
    """RSI classique de Wilder."""
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def compute_atr(high, low, close, period=14):
    """Average True Range."""
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(com=period - 1, min_periods=period).mean()


def add_features(df_sym):
    """Calcule tous les indicateurs pour un symbole."""
    d = df_sym.copy().sort_values('candle_time')
    c = d['close_price']
    h = d['high_price']
    l = d['low_price']
    o = d['open_price']
    v = d['volume']

    # ── TENDANCE ──────────────────────────────────────────
    d['ema9']   = c.ewm(span=9,   adjust=False).mean()
    d['ema21']  = c.ewm(span=21,  adjust=False).mean()
    d['ema50']  = c.ewm(span=50,  adjust=False).mean()
    d['ema200'] = c.ewm(span=200, adjust=False).mean()

    # Position du prix par rapport aux EMAs (en %)
    d['price_vs_ema9']   = (c - d['ema9'])   / d['ema9']   * 100
    d['price_vs_ema21']  = (c - d['ema21'])  / d['ema21']  * 100
    d['price_vs_ema50']  = (c - d['ema50'])  / d['ema50']  * 100
    d['price_vs_ema200'] = (c - d['ema200']) / d['ema200'] * 100

    # Pente des EMAs (direction de la tendance)
    d['ema21_slope']  = d['ema21'].pct_change(3)  * 100
    d['ema50_slope']  = d['ema50'].pct_change(6)  * 100

    # Alignement des EMAs (bullish quand ema9 > ema21 > ema50)
    d['ema_aligned_bull'] = ((d['ema9'] > d['ema21']) & (d['ema21'] > d['ema50'])).astype(int)
    d['ema_aligned_bear'] = ((d['ema9'] < d['ema21']) & (d['ema21'] < d['ema50'])).astype(int)

    # ── MOMENTUM ──────────────────────────────────────────
    d['rsi14'] = compute_rsi(c, 14)
    d['rsi7']  = compute_rsi(c, 7)

    # MACD (12/26/9)
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    d['macd']        = ema12 - ema26
    d['macd_signal'] = d['macd'].ewm(span=9, adjust=False).mean()
    d['macd_hist']   = d['macd'] - d['macd_signal']
    d['macd_cross_bull'] = ((d['macd'] > d['macd_signal']) & (d['macd'].shift(1) <= d['macd_signal'].shift(1))).astype(int)
    d['macd_cross_bear'] = ((d['macd'] < d['macd_signal']) & (d['macd'].shift(1) >= d['macd_signal'].shift(1))).astype(int)

    # Stochastique (14,3)
    low14  = l.rolling(14).min()
    high14 = h.rolling(14).max()
    d['stoch_k'] = (c - low14) / (high14 - low14 + 1e-10) * 100
    d['stoch_d'] = d['stoch_k'].rolling(3).mean()

    # Rendements passés
    d['ret_1']  = c.pct_change(1)  * 100   # 5 min
    d['ret_3']  = c.pct_change(3)  * 100   # 15 min
    d['ret_12'] = c.pct_change(12) * 100   # 1h
    d['ret_48'] = c.pct_change(48) * 100   # 4h
    d['ret_288']= c.pct_change(288)* 100   # 24h

    # ── VOLATILITÉ ────────────────────────────────────────
    d['atr14'] = compute_atr(h, l, c, 14)
    d['atr_pct'] = d['atr14'] / c * 100    # ATR normalisé en %

    # Bollinger Bands (20, 2)
    bb_mid  = c.rolling(20).mean()
    bb_std  = c.rolling(20).std()
    d['bb_upper'] = bb_mid + 2 * bb_std
    d['bb_lower'] = bb_mid - 2 * bb_std
    d['bb_width'] = (d['bb_upper'] - d['bb_lower']) / bb_mid * 100   # largeur en %
    d['bb_pos']   = (c - d['bb_lower']) / (d['bb_upper'] - d['bb_lower'] + 1e-10)  # 0=bas, 1=haut

    # Volatilité réalisée (std des rendements sur 12 périodes)
    d['vol_realized_1h'] = d['ret_1'].rolling(12).std()

    # ── VOLUME ────────────────────────────────────────────
    vol_ma20 = v.rolling(20).mean()
    d['volume_ratio'] = v / (vol_ma20 + 1e-10)  # > 1 = volume anormalement élevé

    # Pression achat/vente (approximation sans tick data)
    d['buy_pressure'] = (c - l) / (h - l + 1e-10)   # 1 = clôture au plus haut
    d['sell_pressure'] = (h - c) / (h - l + 1e-10)  # 1 = clôture au plus bas

    # OBV (On Balance Volume) — tendance du volume
    obv = (np.sign(c.diff()) * v).fillna(0).cumsum()
    d['obv_slope'] = obv.pct_change(12) * 100

    # ── STRUCTURE DE BOUGIE ───────────────────────────────
    body     = (c - o).abs()
    candle_range = (h - l + 1e-10)
    d['body_ratio']      = body / candle_range          # taille corps / range total
    d['upper_wick_ratio']= (h - c.clip(upper=o+body)) / candle_range  # mèche haute
    d['lower_wick_ratio']= (c.clip(lower=o-body) - l) / candle_range  # mèche basse
    d['is_bullish_candle']= (c > o).astype(int)

    # ── CONTEXTE TEMPOREL ─────────────────────────────────
    d['hour_utc']  = d['candle_time'].dt.hour
    d['dow']       = d['candle_time'].dt.dayofweek   # 0=lundi, 6=dimanche
    d['is_weekend']= (d['dow'] >= 5).astype(int)

    # Sessions de trading (approximatives)
    d['session_asia']   = d['hour_utc'].between(0, 8).astype(int)
    d['session_europe'] = d['hour_utc'].between(8, 16).astype(int)
    d['session_us']     = d['hour_utc'].between(13, 22).astype(int)

    # -- REGIME --------------------------------------------

    # Trend strength (ADX)
    def compute_adx(high, low, close, period=14):
        tr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low  - close.shift(1)).abs()
        ], axis=1).max(axis=1)
        atr = tr.ewm(com=period-1, min_periods=period).mean()

        up_move   = high - high.shift(1)
        down_move = low.shift(1) - low

        plus_dm  = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
        minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)

        plus_di  = 100 * pd.Series(plus_dm,  index=close.index).ewm(com=period-1, min_periods=period).mean() / atr
        minus_di = 100 * pd.Series(minus_dm, index=close.index).ewm(com=period-1, min_periods=period).mean() / atr

        dx  = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-10)
        adx = dx.ewm(com=period-1, min_periods=period).mean()
        return adx, plus_di, minus_di

    adx, plus_di, minus_di = compute_adx(h, l, c, 14)
    d['adx_14']      = adx
    d['plus_di']     = plus_di
    d['minus_di']    = minus_di
    d['is_trending'] = (adx > 25).astype(int)
    d['is_ranging']  = (adx < 20).astype(int)

    # Trend direction (EMA200 slope sur plusieurs horizons)
    d['ema200_slope_12']  = d['ema200'].pct_change(12)  * 100   # 1h
    d['ema200_slope_48']  = d['ema200'].pct_change(48)  * 100   # 4h
    d['ema200_slope_288'] = d['ema200'].pct_change(288) * 100   # 24h
    d['trend_direction']  = np.sign(d['ema200_slope_48'])       # +1 haussier, -1 baissier, 0 neutre

    # Volatility regime
    d['atr_ratio_short'] = d['atr14'] / (d['atr14'].rolling(48).mean()  + 1e-10)  # ATR courant vs moyenne 4h
    d['atr_ratio_long']  = d['atr14'] / (d['atr14'].rolling(288).mean() + 1e-10)  # ATR courant vs moyenne 24h

    # Market structure
    d['higher_high_12'] = (h > h.shift(12)).astype(int)   # Higher high sur 1h
    d['lower_low_12']   = (l < l.shift(12)).astype(int)   # Lower low sur 1h
    d['hh_ll_bull']     = ((d['higher_high_12'] == 1) & (d['lower_low_12'] == 0)).astype(int)  # structure haussiere
    d['hh_ll_bear']     = ((d['higher_high_12'] == 0) & (d['lower_low_12'] == 1)).astype(int)  # structure baissiere

    return d


print('Fonctions définies — calcul des features en cours...')

In [ ]:
# === ENRICHISSEMENT MULTI-TIMEFRAME + FEAR & GREED ===

def compute_tf_features(df_5m, df_1h, df_4h, df_fg, symbol):
    sym_1h = df_1h[df_1h['symbol'] == symbol].copy().sort_values('candle_time')
    sym_4h = df_4h[df_4h['symbol'] == symbol].copy().sort_values('candle_time')
    d5 = df_5m[df_5m['symbol'] == symbol].copy().sort_values('candle_time')

    # -- Features 1h ----------------------------------------------------------
    if len(sym_1h) >= 200:
        c1 = sym_1h['close_price']
        sym_1h['rsi_1h']           = compute_rsi(c1, 14)
        sym_1h['ema200_1h']        = c1.ewm(span=200, adjust=False).mean()
        sym_1h['ema50_1h']         = c1.ewm(span=50,  adjust=False).mean()
        sym_1h['atr_1h']           = compute_atr(sym_1h['high_price'],
                                                   sym_1h['low_price'], c1, 14)
        vol_ma = sym_1h['volume'].rolling(20).mean()
        sym_1h['volume_zscore_1h'] = (sym_1h['volume'] - vol_ma) / (vol_ma + 1e-10)
        sym_1h['ema200_slope_1h']  = sym_1h['ema200_1h'].pct_change(4) * 100
        feat_1h = sym_1h[['candle_time','rsi_1h','ema200_1h','ema50_1h',
                           'atr_1h','volume_zscore_1h','ema200_slope_1h']]
        d5 = pd.merge_asof(d5.sort_values('candle_time'),
                           feat_1h.sort_values('candle_time'),
                           on='candle_time', direction='backward')
    else:
        # Fallback 5m inline
        c5 = d5['close_price']
        d5['rsi_1h']           = compute_rsi(c5, 14)
        d5['ema200_1h']        = c5.ewm(span=200, adjust=False).mean()
        d5['ema50_1h']         = c5.ewm(span=50,  adjust=False).mean()
        d5['atr_1h']           = compute_atr(d5['high_price'], d5['low_price'], c5, 14)
        vol_ma = d5['volume'].rolling(20).mean()
        d5['volume_zscore_1h'] = (d5['volume'] - vol_ma) / (vol_ma + 1e-10)
        d5['ema200_slope_1h']  = d5['ema200_1h'].pct_change(48) * 100

    # -- Features 4h ----------------------------------------------------------
    if len(sym_4h) >= 200:
        c4 = sym_4h['close_price']
        sym_4h['rsi_4h']          = compute_rsi(c4, 14)
        sym_4h['ema200_4h']       = c4.ewm(span=200, adjust=False).mean()
        sym_4h['ema50_4h']        = c4.ewm(span=50,  adjust=False).mean()
        sym_4h['ema200_slope_4h'] = sym_4h['ema200_4h'].pct_change(6) * 100
        feat_4h = sym_4h[['candle_time','rsi_4h','ema200_4h','ema50_4h',
                           'ema200_slope_4h']]
        d5 = pd.merge_asof(d5.sort_values('candle_time'),
                           feat_4h.sort_values('candle_time'),
                           on='candle_time', direction='backward')
    elif len(sym_1h) >= 200:
        # Proxy 1h -> 4h
        d5['rsi_4h']          = d5['rsi_1h']
        d5['ema200_4h']       = d5['ema200_1h']
        d5['ema50_4h']        = d5['ema50_1h']
        d5['ema200_slope_4h'] = d5['ema200_slope_1h']
    else:
        # Fallback 5m inline
        c5 = d5['close_price']
        d5['rsi_4h']          = compute_rsi(c5, 14)
        d5['ema200_4h']       = c5.ewm(span=200, adjust=False).mean()
        d5['ema50_4h']        = c5.ewm(span=50,  adjust=False).mean()
        d5['ema200_slope_4h'] = d5['ema200_4h'].pct_change(48) * 100

    # -- Trend alignment ------------------------------------------------------
    d5['trend_bull_1h']      = (d5['ema200_slope_1h'] > 0).astype(int)
    d5['trend_bull_4h']      = (d5['ema200_slope_4h'] > 0).astype(int)
    d5['trend_aligned_bull'] = ((d5['trend_bull_1h'] == 1) &
                                 (d5['trend_bull_4h'] == 1)).astype(int)
    d5['above_ema200_1h']    = (d5['close_price'] > d5['ema200_1h']).astype(int)
    d5['above_ema200_4h']    = (d5['close_price'] > d5['ema200_4h']).astype(int)

    # -- Fear & Greed (daily -> joint par date) -------------------------------
    d5['date_utc'] = d5['candle_time'].dt.floor('1D')
    d5 = pd.merge_asof(
        d5.sort_values('candle_time'),
        df_fg[['date','fear_greed_value']].sort_values('date'),
        left_on='date_utc',
        right_on='date',
        direction='backward'
    )
    d5 = d5.drop(columns=['date_utc','date'], errors='ignore')
    d5['fear_greed_value'] = d5['fear_greed_value'].fillna(50)

    return d5.sort_values('candle_time').reset_index(drop=True)


# Appliquer par symbole
print('Enrichissement MTF + Fear & Greed...')
dfs_enriched = []
for symbol in df['symbol'].unique():
    df_sym = compute_tf_features(df, df_1h, df_4h, df_fg, symbol)
    dfs_enriched.append(df_sym)
    print(f'  {symbol}: {len(df_sym):,} lignes, {df_sym.shape[1]} colonnes')

df = pd.concat(dfs_enriched, ignore_index=True)
print(f'\nDataset enrichi total : {len(df):,} lignes x {df.shape[1]} colonnes')

MTF_FEATURES = ['rsi_1h','ema200_1h','ema50_1h','atr_1h','volume_zscore_1h',
                'ema200_slope_1h','rsi_4h','ema200_4h','ema50_4h',
                'ema200_slope_4h','trend_bull_1h','trend_bull_4h',
                'trend_aligned_bull','above_ema200_1h','above_ema200_4h',
                'fear_greed_value']
print(f'MTF features : {len(MTF_FEATURES)} | dont F&G : fear_greed_value (daily)')


In [ ]:
# Calcul sur TOUT le dataset (train + test)
# Les features sont calculees par symbole independamment

parts = []
for sym in SYMBOLS_ALL:
    sub = df[df['symbol'] == sym].copy()
    sub_feat = add_features(sub)
    parts.append(sub_feat)
    print(f'  {sym} : {len(sub_feat):,} bougies, {sub_feat.shape[1]} colonnes')

df_feat = pd.concat(parts, ignore_index=True)
df_feat = df_feat.sort_values(['symbol', 'candle_time']).reset_index(drop=True)

print(f'\nDataset enrichi : {df_feat.shape[0]:,} lignes x {df_feat.shape[1]} colonnes')

# Features calculees — auto-detect (on exclut les colonnes non-feature)
feature_cols = [c for c in df_feat.columns if c not in
                ['symbol', 'candle_time', 'open_price', 'high_price',
                 'low_price', 'close_price', 'volume', 'snapshot_count']]

# Nouvelles features de regime injectees dans add_features
REGIME_FEATURES = [
    'adx_14', 'plus_di', 'minus_di', 'is_trending', 'is_ranging',
    'ema200_slope_12', 'ema200_slope_48', 'ema200_slope_288', 'trend_direction',
    'atr_ratio_short', 'atr_ratio_long',
    'higher_high_12', 'lower_low_12', 'hh_ll_bull', 'hh_ll_bear',
]
baseline_cols = [c for c in feature_cols if c not in REGIME_FEATURES]
print(f'Features baseline : {len(baseline_cols)}')
print(f'Features regime   : {len(REGIME_FEATURES)}')
print(f'Features total    : {len(feature_cols)}')

# Diagnostic NaN : rolling(288) = 24h de warmup par symbole
n_nan_any = df_feat[feature_cols].isna().any(axis=1).sum()
print(f'\nLignes avec >= 1 NaN sur features : {n_nan_any:,} / {len(df_feat):,} '
      f'({n_nan_any/len(df_feat)*100:.2f} %)  [attendu ~288/symbole]')
top_nan = df_feat[feature_cols].isna().sum().sort_values(ascending=False).head(5)
print('Top 5 features avec le plus de NaN (warmup) :')
print(top_nan.to_string())

print('\nListe complete feature_cols :')
print(feature_cols)

## 4. Labellisation des trades

Pour chaque bougie, on simule un BUY et on regarde ce qui se passe dans les 12 bougies suivantes (1h max).

- **TP** : +0.7% → **label = 1 (WIN)**
- **SL** : -0.5% → **label = 0 (LOSS)**  
- **Ni l'un ni l'autre en 1h** → **label = 0 (LOSS)**
- **Frais A/R** : 0.3% déduits du TP (net = +0.4%)

> On labellise le fichier complet, mais on ne regardera les labels que sur le TRAIN set pendant l'EDA.

In [ ]:
def label_trades(df_sym, tp_pct, sl_pct, fees_pct, horizon):
    """
    Pour chaque bougie t, simule un BUY au close_price[t].
    Regarde les horizon bougies suivantes.
    Retourne label=1 si TP touché avant SL, 0 sinon.
    """
    d = df_sym.sort_values('candle_time').reset_index(drop=True)
    closes = d['close_price'].values
    highs  = d['high_price'].values
    lows   = d['low_price'].values
    n = len(d)

    labels   = np.zeros(n, dtype=int)
    outcomes = ['timeout'] * n
    exit_bar = np.full(n, -1, dtype=int)
    exit_ret = np.zeros(n)

    for i in range(n - horizon):
        entry = closes[i]
        tp    = entry * (1 + tp_pct)
        sl    = entry * (1 - sl_pct)

        for j in range(1, horizon + 1):
            if lows[i+j] <= sl:
                outcomes[i] = 'loss'
                exit_bar[i] = j
                exit_ret[i] = -sl_pct - fees_pct
                break
            if highs[i+j] >= tp:
                outcomes[i] = 'win'
                labels[i]   = 1
                exit_bar[i] = j
                exit_ret[i] = tp_pct - fees_pct
                break

    d['label']    = labels
    d['outcome']  = outcomes
    d['exit_bar'] = exit_bar
    d['exit_ret'] = exit_ret
    return d


print('Labellisation en cours...')
parts2 = []
for sym in SYMBOLS_ALL:
    sub = df_feat[df_feat['symbol'] == sym].copy()
    sub_labelled = label_trades(sub, tp_pct=TP_PCT, sl_pct=SL_PCT, fees_pct=FEES_PCT, horizon=HORIZON)
    win_rate = sub_labelled['label'].mean() * 100
    print(f'  {sym:12s}: win rate brut = {win_rate:.1f}%')
    parts2.append(sub_labelled)

df_labelled = pd.concat(parts2, ignore_index=True)
df_labelled = df_labelled.sort_values(['symbol', 'candle_time']).reset_index(drop=True)

print(f'\nWin rate global : {df_labelled["label"].mean()*100:.1f}%')
print(f'Distribution    : {df_labelled["outcome"].value_counts().to_dict()}')

## 5. Split final train / test

In [ ]:
split_ts = pd.Timestamp(SPLIT_DATE, tz='UTC')

# ============================================================
#  EMBARGO / PURGE — eviter la fuite a la frontiere train/test
#  Le label regarde HORIZON bougies dans le futur. Les dernieres
#  bougies du TRAIN avant la coupure ont donc un label calcule
#  avec des prix qui tombent dans le TEST. On les retire (par symbole).
# ============================================================
EMBARGO = HORIZON   # 12 bougies = 1h de zone tampon (= horizon du label)

train_parts, test_parts, n_purged = [], [], 0
for sym, g in df_labelled.groupby('symbol'):
    g = g.sort_values('candle_time')
    g_train = g[g['candle_time'] <  split_ts]
    g_test  = g[g['candle_time'] >= split_ts]
    if len(g_train) > EMBARGO:
        n_purged += EMBARGO
        g_train = g_train.iloc[:-EMBARGO]
    else:
        n_purged += len(g_train)
        g_train = g_train.iloc[0:0]
    train_parts.append(g_train)
    test_parts.append(g_test)

train_df = pd.concat(train_parts).copy()
test_df  = pd.concat(test_parts).copy()

# Supprimer les lignes avec NaN dans les features (dues aux rolling windows au debut)
train_df = train_df.dropna(subset=feature_cols)
test_df  = test_df.dropna(subset=feature_cols)

print(f'Embargo applique : {EMBARGO} bougies retirees par symbole ({n_purged:,} au total)')
print(f'TRAIN : {len(train_df):,} bougies | win rate : {train_df["label"].mean()*100:.1f}%')
print(f'TEST  : {len(test_df):,}  bougies | win rate : {test_df["label"].mean()*100:.1f}%')

print()
print('Win rate par symbole (TRAIN) :')
for sym, g in train_df.groupby('symbol'):
    print(f'  {sym:12s}: {g["label"].mean()*100:.1f}%  ({len(g):,} bougies)')

## 6. Vérification rapide des features clés (sur TRAIN uniquement)

In [ ]:
# Distribution des features les plus importantes visuellement
key_features = ['rsi14', 'price_vs_ema50', 'bb_pos', 'atr_pct', 'volume_ratio', 'macd_hist']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Distribution des features clés — WIN vs LOSS (TRAIN)', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.flatten(), key_features):
    wins   = train_df[train_df['label'] == 1][feat].dropna()
    losses = train_df[train_df['label'] == 0][feat].dropna()
    
    # Clip pour visualisation
    p1, p99 = train_df[feat].quantile([0.01, 0.99])
    wins_c   = wins.clip(p1, p99)
    losses_c = losses.clip(p1, p99)
    
    ax.hist(losses_c, bins=50, alpha=0.5, color='tomato',  label='LOSS', density=True)
    ax.hist(wins_c,   bins=50, alpha=0.5, color='steelblue', label='WIN',  density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Corrélation features vs label (sur TRAIN)
corr = train_df[feature_cols + ['label']].corr()['label'].drop('label').sort_values()

fig, ax = plt.subplots(figsize=(8, 14))
colors = ['tomato' if x < 0 else 'steelblue' for x in corr]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Corrélation features → label WIN (TRAIN)', fontsize=13)
ax.set_xlabel('Corrélation de Pearson')
plt.tight_layout()
plt.show()

print('\nTop 10 features positives :')
print(corr.tail(10).to_string())
print('\nTop 10 features négatives :')
print(corr.head(10).to_string())

In [ ]:
# Ajoute ça AVANT la cellule de sauvegarde
import shutil
shutil.copy(OUTPUT_PATH + 'train_features.csv', OUTPUT_PATH + 'train_features_tp07.csv')
shutil.copy(OUTPUT_PATH + 'test_features.csv',  OUTPUT_PATH + 'test_features_tp07.csv')
shutil.copy(OUTPUT_PATH + 'config.json',         OUTPUT_PATH + 'config_tp07.json')
print('Backup OK')

## 7. Sauvegarde

In [ ]:
import os
import json
from datetime import datetime

os.makedirs(OUTPUT_PATH, exist_ok=True)
DATE = datetime.now().strftime('%Y%m%d')

# Sauvegarde versionnee
train_df.to_csv(OUTPUT_PATH + f'train_features_{DATE}.csv', index=False)
test_df.to_csv(OUTPUT_PATH  + f'test_features_{DATE}.csv',  index=False)
# Copie sans suffixe (compatibilite Railway + fallback)
train_df.to_csv(OUTPUT_PATH + 'train_features.csv', index=False)
test_df.to_csv(OUTPUT_PATH  + 'test_features.csv',  index=False)

with open(OUTPUT_PATH + f'feature_cols_{DATE}.json', 'w') as f:
    json.dump(feature_cols, f)
with open(OUTPUT_PATH + 'feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

config = {
    'tp_pct'    : TP_PCT,
    'sl_pct'    : SL_PCT,
    'fees_pct'  : FEES_PCT,
    'horizon'   : HORIZON,
    'tp_net'    : TP_NET,
    'sl_net'    : SL_NET,
    'breakeven' : BREAKEVEN,
    'split_date': SPLIT_DATE
}
with open(OUTPUT_PATH + f'config_{DATE}.json', 'w') as f:
    json.dump(config, f, indent=2)
with open(OUTPUT_PATH + 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

# latest.json : pointe sur les derniers fichiers en date
latest = {
    'train': f'train_features_{DATE}.csv',
    'test' : f'test_features_{DATE}.csv',
    'feature_cols': f'feature_cols_{DATE}.json',
    'config': f'config_{DATE}.json',
    'date': DATE
}
with open(OUTPUT_PATH + 'latest.json', 'w') as f:
    json.dump(latest, f, indent=2)

print(f'Sauvegarde dans {OUTPUT_PATH}')
print(f'  train_features_{DATE}.csv : {len(train_df):,} lignes')
print(f'  test_features_{DATE}.csv  : {len(test_df):,} lignes')
print(f'  feature_cols_{DATE}.json  : {len(feature_cols)} features')
print(f'  config_{DATE}.json        : TP={TP_PCT*100}% SL={SL_PCT*100}% fees={FEES_PCT*100}%')
print(f'latest.json mis a jour -> {latest}')
print()
print('Notebook 01 termine. Passe au notebook 02_eda.ipynb')

In [ ]:
import os, json

files = ['train_features.csv', 'test_features.csv', 'config.json']
for f in files:
    path = OUTPUT_PATH + f
    size = os.path.getsize(path)
    print(f'{f} : {size/1024/1024:.1f} MB')

# Vérifie que la config est correcte
with open(OUTPUT_PATH + 'config.json') as f:
    cfg = json.load(f)
print()
print(f'TP_PCT   : {cfg["tp_pct"]*100:.4f}%')
print(f'SL_PCT   : {cfg["sl_pct"]*100:.4f}%')
print(f'FEES_PCT : {cfg["fees_pct"]*100:.4f}%')
print(f'TP net   : {cfg["tp_net"]*100:.4f}%')
print(f'SL net   : {cfg["sl_net"]*100:.4f}%')
print(f'Breakeven: {cfg["breakeven"]*100:.2f}%')